# Machine Learning with Tabular Data: Diabetes Readmission Prediction

Predict discharge-time readmission status for encounters involving patients with diabetes:

- `NO`: no recorded readmission
- `>30`: readmission after more than 30 days
- `<30`: readmission within 30 days

**Workflow:** patient-disjoint splitting -> train-only EDA and preprocessing -> grouped model selection -> held-out test evaluation.


# Phase 1: Data Acquisition and Initial Inspection

## 1.1 Download and Extract the Dataset

In [ ]:
import os
import sys
import subprocess
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Root-level project setup for local execution and Google Colab.
REPO_URL = "https://github.com/nam21-12-06/machine-learning-assignment.git"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    print("Running on Google Colab")
    PROJECT_DIR = Path("/content/machine-learning-assignment")
    if not PROJECT_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR),
            ],
            check=True,
        )
    else:
        print("Updating repository to latest commit...")
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "pull", "origin", REPO_BRANCH],
            check=False,
        )
else:
    print("Running on Local Machine")
    PROJECT_DIR = Path.cwd().resolve()
    if PROJECT_DIR.name == "notebooks":
        PROJECT_DIR = PROJECT_DIR.parent
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
MODULES_DIR = PROJECT_DIR / "modules"
REPORTS_DIR = PROJECT_DIR / "reports"
RESULTS_DIR = PROJECT_DIR / "results"
FEATURES_DIR = PROJECT_DIR / "features"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [DATA_DIR, RAW_DIR, REPORTS_DIR, RESULTS_DIR, FEATURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Route experiment reports, tables and figures to results directory
REPORTS_DIR = RESULTS_DIR

required_modules = [
    MODULES_DIR / "preprocessing.py",
    MODULES_DIR / "svm_model.py",
    MODULES_DIR / "evaluation.py",
    MODULES_DIR / "logistic_regression_model.py",
    MODULES_DIR / "random_forest_model.py",
    MODULES_DIR / "mlp_model.py",
]
missing_modules = [path.name for path in required_modules if not path.exists()]
if missing_modules:
    raise FileNotFoundError(
        "Required project modules are missing: " + ", ".join(missing_modules)
    )

print(f"Project directory set to: {PROJECT_DIR}")
print("Supporting modules and output directories are ready.")

In [ ]:
# Dowload dataset
DATA_URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"

ZIP_PATH = DATA_DIR / "diabetes.zip"

if not ZIP_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Download completed.")
else:
    print("Dataset archive already exists. Skipping download.")

In [ ]:
# Extract dataset
if not list(RAW_DIR.glob("*.csv")):
    print("Extracting files...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Extraction completed.")
else:
    print("Files already extracted.")
print("\nFiles in raw directory:")
for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(f" - {path.name}")

## 1.2 Load the Dataset

Missing values are parsed explicitly; the laboratory state `None` is retained as `NotMeasured` rather than treated as missing data.


In [ ]:
TARGET = "readmitted"
CLASS_ORDER = ["NO", ">30", "<30"]
LAB_COLS = ["max_glu_serum", "A1Cresult"]
IDENTIFIER_COLS = ["encounter_id", "patient_nbr"]
CODE_COLS = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]
DIAG_COLS = ["diag_1", "diag_2", "diag_3"]

data_paths = list(RAW_DIR.rglob("diabetic_data.csv"))
assert len(data_paths) == 1, "Expected exactly one diabetic_data.csv file."
DATA_PATH = data_paths[0]

# encounter_id remains numeric. The other identifiers/codes are categorical strings.
categorical_string_cols = ["patient_nbr"] + CODE_COLS + DIAG_COLS
df_raw = pd.read_csv(
    DATA_PATH,
    keep_default_na=False,
    na_values=["?", ""],
    dtype={column: object for column in categorical_string_cols},
    low_memory=False,
)

df = df_raw.copy()

# Strip whitespace without converting genuine NaN values into the string "nan".
for column in df.select_dtypes(include="object"):
    df[column] = df[column].str.strip()

# "None" means the test was not ordered/performed, not an unknown data value.
for column in LAB_COLS:
    df[column] = df[column].replace({"None": "NotMeasured"})

assert df[TARGET].notna().all(), "The target contains missing values."
assert set(df[TARGET].unique()) <= set(CLASS_ORDER), "Unexpected target class found."
assert df[IDENTIFIER_COLS].notna().all().all(), "An identifier is missing."

print("Dataset shape:", df.shape)
display(df[LAB_COLS].head())


In [ ]:
print(f"Dataset shape:{df.shape}") # 50 features and 101766 samples
print(f"Number of rows:{df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Columns names
print("COLUMN NAMES")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

In [ ]:
# Print first 5 rows
display(df.head())

In [ ]:
# Data type
display(df.dtypes.to_frame(name="dtype"))

## 1.3 Initial Data Quality Assessment

Fully duplicated rows are assessed separately from valid repeated encounters belonging to the same patient.


In [ ]:
print("=" * 60)
print("DATASET OVERVIEW - NUMERICAL & CATEGORICAL")
print("=" * 60)

# Keep encounter_id numeric so chronological sorting remains numeric later.
# Categorical ID columns are converted to strings to avoid treating them as measurements.
id_cols = ['patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']
df[id_cols] = df[id_cols].astype(str)

# Observe overall distribution
display(df.describe(include="all").T)

## 1.4 Missing Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

print("Missing information:")
display(missing_summary.query("missing_count > 0").round(2))

lab_status_summary = pd.DataFrame({
    "not_measured_count": [
        df[col].eq("NotMeasured").sum() for col in LAB_COLS
    ],
    "not_measured_pct": [
        df[col].eq("NotMeasured").mean() * 100 for col in LAB_COLS
    ],
    "missing_count": [
        df[col].isna().sum() for col in LAB_COLS
    ],
}, index=LAB_COLS)

print("Laboratory test availability:")
display(lab_status_summary.round(2))

visits_per_patient = df.groupby("patient_nbr").size()

print("Fully duplicated rows:", df.duplicated().sum())
print("Repeated encounter IDs:", df["encounter_id"].duplicated().sum())
print("Unique patients:", visits_per_patient.size)
print("Patients with multiple encounters:", visits_per_patient.gt(1).sum())
print("Encounters beyond one per patient:", (visits_per_patient - 1).sum())

display(visits_per_patient.describe().to_frame("encounters_per_patient"))

## 1.5 Duplicate Analysis

In [ ]:
print("=" * 60)
print("DUPLICATE & PATIENT ENCOUNTER ANALYSIS")
print("=" * 60)

# Check for completely duplicated rows
duplicate_count = df.duplicated().sum()
print(f"Total fully duplicated rows: {duplicate_count}")

# BONUS: Check if a single patient has multiple encounters (readmissions)
unique_patients = df['patient_nbr'].nunique()
total_encounters = len(df)
multiple_encounters = total_encounters - unique_patients

print(f"Total encounters (rows): {total_encounters:,}")
print(f"Total unique patients: {unique_patients:,}")
print(f"Number of return visits (multiple encounters): {multiple_encounters:,}")


## 1.6 Target Distribution

In [ ]:
TARGET = "readmitted"

# Count each target class
target_counts = df[TARGET].value_counts()

# Calculate percentages
target_percentages = df[TARGET].value_counts(normalize=True) * 100

# Combine into one summary table
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

print("=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

display(target_summary)

In [ ]:
# Visualization
plt.figure(figsize=(9, 6))
# Use 'viridis' palette for better aesthetic, assign x to hue to avoid warnings

ax = sns.countplot(data=df, x=TARGET, order=target_counts.index, hue=TARGET, palette='viridis', legend=False)
plt.title("Distribution of Readmission Classes (Imbalanced)", fontsize=14, fontweight='bold')
plt.xlabel("Readmission Status", fontsize=12)
plt.ylabel("Number of Encounters", fontsize=12)

# Add both count and percentage labels on top of the bars
for i, container in enumerate(ax.containers):
    labels = [f'{v.get_height():,} \n({target_percentages.iloc[j]:.1f}%)' for j, v in enumerate(container)]
    ax.bar_label(container, labels=labels, padding=5, fontweight='bold')

# Expand y-axis slightly to fit the tall labels
plt.ylim(0, target_counts.max() * 1.2)
plt.tight_layout()
plt.show()

# Phase 2: Cohort Definition and Patient-Level Holdout Split

Death/hospice encounters are excluded, and all remaining encounters are split by `patient_nbr` to prevent patient overlap.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

DEAD_HOSPICE_IDS = {"11", "13", "14", "19", "20", "21"}

df_cohort = df.loc[
    ~df["discharge_disposition_id"].isin(DEAD_HOSPICE_IDS)
].copy()

print("Original encounters:", f"{len(df):,}")
print("Eligible encounters:", f"{len(df_cohort):,}")
print("Excluded death/hospice encounters:", f"{len(df) - len(df_cohort):,}")

overall_distribution = (
    df_cohort[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0)
)

splitter = GroupShuffleSplit(n_splits=20, test_size=0.20, random_state=RANDOM_STATE)
split_candidates = []
for candidate_id, (candidate_train, candidate_test) in enumerate(
    splitter.split(df_cohort, df_cohort[TARGET], groups=df_cohort["patient_nbr"])
):
    test_distribution = (
        df_cohort.iloc[candidate_test][TARGET]
        .value_counts(normalize=True)
        .reindex(CLASS_ORDER, fill_value=0)
    )
    class_gap = (test_distribution - overall_distribution).abs().sum()
    size_gap = abs(len(candidate_test) / len(df_cohort) - 0.20)
    split_candidates.append((class_gap + size_gap, candidate_id, candidate_train, candidate_test))

_, selected_candidate, train_positions, test_positions = min(
    split_candidates, key=lambda item: item[0]
)

df_train_raw = df_cohort.iloc[train_positions].copy()
df_test_raw = df_cohort.iloc[test_positions].copy()

train_patients = set(df_train_raw["patient_nbr"])
test_patients = set(df_test_raw["patient_nbr"])
patient_overlap = train_patients.intersection(test_patients)
assert not patient_overlap, "Patient leakage detected between train and test."

split_distribution = pd.concat({
    "cohort_pct": overall_distribution * 100,
    "train_pct": df_train_raw[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
    "test_pct": df_test_raw[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
}, axis=1)

print("Selected GroupShuffleSplit candidate:", selected_candidate)
print("Train encounters / patients:", f"{len(df_train_raw):,}", f"/ {len(train_patients):,}")
print("Test encounters / patients:", f"{len(df_test_raw):,}", f"/ {len(test_patients):,}")
print("Patient overlap:", len(patient_overlap))
display(split_distribution.round(2))

# All detailed EDA below uses only the training partition.
eda_df = df_train_raw


# Phase 3: Exploratory Data Analysis on the Training Set

## 3.1 Categorical Features


In [ ]:
TARGET = "readmitted"

print("=" * 60)
print("TRAINING-SET CATEGORICAL SUMMARY")
print("=" * 60)

categorical_cols = eda_df.select_dtypes(include=["object", "string"]).columns.tolist()
excluded_cols = [TARGET, "encounter_id", "patient_nbr"]
categorical_features_eda = [c for c in categorical_cols if c not in excluded_cols]

cat_summary_rows = []
for column in categorical_features_eda:
    values = eda_df[column].fillna("Missing")
    counts = values.value_counts()
    cat_summary_rows.append({
        "feature": column,
        "unique_values": values.nunique(),
        "dominant_value": counts.index[0],
        "dominant_pct": counts.iloc[0] / len(eda_df) * 100,
    })

cat_summary = pd.DataFrame(cat_summary_rows).sort_values("dominant_pct", ascending=False)
display(cat_summary.round(2))

near_constant_eda = cat_summary.query("dominant_pct > 95")
print(f"Near-constant candidates (>95% dominant): {len(near_constant_eda)}")
print("These are candidates for validation, not automatically useless features.")


In [ ]:
print("=" * 60)
print("DISTRIBUTION OF KEY CATEGORICAL FEATURES - TRAINING SET")
print("=" * 60)

key_categorical_features = ["gender", "race", "age", "change", "diabetesMed", "insulin"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, column in zip(axes, key_categorical_features):
    plot_data = eda_df[column].fillna("Missing").astype(str)
    sns.countplot(
        x=plot_data,
        order=plot_data.value_counts().index,
        hue=plot_data,
        palette="Set2",
        legend=False,
        ax=axis,
    )
    axis.set_title(f"Distribution of {column}", fontsize=13, fontweight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Number of encounters")
    axis.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 3.2 Numerical Features and Outliers


In [ ]:
print("=" * 60)
print("NUMERICAL DISTRIBUTIONS AND OUTLIERS - TRAINING SET")
print("=" * 60)

numerical_cols = [
    column for column in eda_df.select_dtypes(include=["int64", "float64"]).columns
    if column != "encounter_id"
]

numeric_summary_rows = []
for column in numerical_cols:
    values = eda_df[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_mask = (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)
    numeric_summary_rows.append({
        "feature": column,
        "skewness": values.skew(),
        "iqr_outlier_pct": outlier_mask.mean() * 100,
        "median": values.median(),
    })

numeric_summary = pd.DataFrame(numeric_summary_rows).sort_values(
    "skewness", key=lambda series: series.abs(), ascending=False
)
display(numeric_summary.round(2))

fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 4 * len(numerical_cols)))
for row, column in enumerate(numerical_cols):
    sns.histplot(eda_df[column], kde=True, ax=axes[row, 0], color="steelblue")
    axes[row, 0].set_title(f"Distribution of {column}", fontweight="bold")
    sns.boxplot(x=eda_df[column], ax=axes[row, 1], color="mediumseagreen")
    axes[row, 1].set_title(f"Boxplot of {column}", fontweight="bold")

plt.tight_layout()
plt.show()


## 3.3 Numerical Correlation and Bivariate Analysis


In [ ]:
print("=" * 60)
print("NUMERICAL CORRELATION - TRAINING SET")
print("=" * 60)

corr_matrix = eda_df[numerical_cols].corr()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top_correlations = (
    upper_triangle.stack()
    .rename("correlation")
    .to_frame()
    .assign(abs_correlation=lambda table: table["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
    .head(10)
)

print("Ten strongest pairwise correlations by absolute value:")
display(top_correlations.round(3))

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Correlation Heatmap of Numerical Features (Training Set)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
print("=" * 60)
print("BIVARIATE ANALYSIS - TRAINING SET")
print("=" * 60)

time_summary = eda_df.groupby(TARGET, observed=False)["time_in_hospital"].agg(
    count="size", mean="mean", median="median", std="std"
).reindex(CLASS_ORDER)
print("Time in hospital by target class:")
display(time_summary.round(2))

def target_rate_table(data, feature):
    values = data[feature].fillna("Missing")
    table = pd.crosstab(values, data[TARGET], normalize="index")
    return (table.reindex(columns=CLASS_ORDER, fill_value=0) * 100).round(2)

for feature in ["age", "gender", "race", "insulin", "A1Cresult"]:
    print(f"Target percentages within each {feature} group:")
    display(target_rate_table(eda_df, feature))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=eda_df, x=TARGET, y="time_in_hospital", order=CLASS_ORDER,
            color="mediumseagreen", ax=axes[0])
axes[0].set_title("Time in Hospital vs Readmission Status")

sns.countplot(data=eda_df, x="age", hue=TARGET, hue_order=CLASS_ORDER,
              palette="viridis", ax=axes[1])
axes[1].set_title("Age Group vs Readmission Status")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


# Phase 4: Data Preprocessing


## 4.1 Initialize the Patient-Disjoint Splits


In [ ]:
print("=" * 60)
print("INITIALIZE CLEAN TRAIN AND TEST FRAMES")
print("=" * 60)

df_train_clean = df_train_raw.copy()
df_test_clean = df_test_raw.copy()

print(f"Training encounters: {len(df_train_clean):,}")
print(f"Test encounters: {len(df_test_clean):,}")
print("Repeated encounters are retained; patient groups remain disjoint.")


## 4.2 Remove Non-Predictive and Unusable Columns


In [ ]:
print("=" * 60)
print("DROP NON-PREDICTIVE AND HEAVILY MISSING COLUMNS")
print("=" * 60)

# patient_nbr is deliberately retained until X/y/groups are separated.
columns_to_drop = ["encounter_id", "weight"]
df_train_clean = df_train_clean.drop(columns=columns_to_drop, errors="ignore")
df_test_clean = df_test_clean.drop(columns=columns_to_drop, errors="ignore")

print("Dropped from both splits:", columns_to_drop)
print("patient_nbr retained temporarily for grouped validation.")
print("Train shape:", df_train_clean.shape)
print("Test shape:", df_test_clean.shape)


In [ ]:
print("=" * 60)
print("LOW-VARIANCE AUDIT ON TRAINING DATA")
print("=" * 60)

audit_features = [
    column for column in df_train_clean.columns
    if column not in {TARGET, "patient_nbr"}
]

variance_audit_rows = []
for column in audit_features:
    counts = df_train_clean[column].value_counts(dropna=False)
    variance_audit_rows.append({
        "feature": column,
        "unique_values": df_train_clean[column].nunique(dropna=False),
        "dominant_pct": counts.iloc[0] / len(df_train_clean) * 100,
    })

variance_audit = pd.DataFrame(variance_audit_rows).sort_values("dominant_pct", ascending=False)
constant_candidates = variance_audit.query("unique_values <= 1")["feature"].tolist()
near_constant_candidates = variance_audit.query(
    "unique_values > 1 and dominant_pct > 95"
)["feature"].tolist()

display(variance_audit.head(25).round(2))
print("Zero-variance columns to drop:", constant_candidates)
print("Near-constant columns retained for comparison:", near_constant_candidates)


In [ ]:
print("=" * 60)
print("CURRENT STATE BEFORE MISSING-CODE STANDARDIZATION")
print("=" * 60)

print(f"Training data: {df_train_clean.shape[0]:,} rows, {df_train_clean.shape[1]} columns")
print(f"Test data:     {df_test_clean.shape[0]:,} rows, {df_test_clean.shape[1]} columns")

train_missing = pd.DataFrame({
    "missing_count": df_train_clean.isna().sum(),
    "missing_pct": df_train_clean.isna().mean() * 100,
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

display(train_missing.round(2))


## 4.3 Standardize Hidden Missing Codes


In [ ]:
print("STANDARDIZING HIDDEN MISSING CODES")

invalid_code_map = {
    "admission_type_id": ["5", "6", "8"],
    "discharge_disposition_id": ["18", "25", "26"],
    "admission_source_id": ["9", "15", "17", "20", "21"],
}

def standardize_hidden_missing(frame):
    result = frame.copy()
    for column, invalid_codes in invalid_code_map.items():
        result[column] = result[column].replace(invalid_codes, np.nan)
    return result

df_train_clean = standardize_hidden_missing(df_train_clean)
df_test_clean = standardize_hidden_missing(df_test_clean)


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_train_clean.isna().sum(),
    "missing_pct": df_train_clean.isna().mean() * 100,
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

print("Training-set missing values after hidden-code standardization:")
display(missing_summary.round(2))

print("Laboratory states are retained separately from genuine missing values:")
display(pd.DataFrame({
    "not_measured_pct": [df_train_clean[c].eq("NotMeasured").mean() * 100 for c in LAB_COLS],
    "missing_pct": [df_train_clean[c].isna().mean() * 100 for c in LAB_COLS],
}, index=LAB_COLS).round(2))

print("No imputer is fitted here; imputation remains inside the sklearn pipeline.")


## 4.4 Feature Engineering: Age and ICD-9


In [ ]:
print("=" * 60)
print("CONFIGURABLE FEATURE ENGINEERING")
print("=" * 60)

AGE_ENCODING = "onehot"  # Alternative experiment: "midpoint"

age_mapping = {
    "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35,
    "[40-50)": 45, "[50-60)": 55, "[60-70)": 65, "[70-80)": 75,
    "[80-90)": 85, "[90-100)": 95,
}

def map_icd9_to_category(value):
    if pd.isna(value):
        return np.nan

    code = str(value).upper()
    if code.startswith("250"):
        return "Diabetes"
    if code.startswith(("V", "E")):
        return "Supplemental"

    try:
        number = float(code)
    except ValueError:
        return "Other"

    if 390 <= number <= 459 or number == 785:
        return "Circulatory"
    if 460 <= number <= 519 or number == 786:
        return "Respiratory"
    if 520 <= number <= 579 or number == 787:
        return "Digestive"
    if 800 <= number <= 999:
        return "Injury"
    if 710 <= number <= 739:
        return "Musculoskeletal"
    if 580 <= number <= 629 or number == 788:
        return "Genitourinary"
    if 140 <= number <= 239:
        return "Neoplasms"
    return "Other"

def apply_feature_engineering(frame, age_encoding=AGE_ENCODING):
    result = frame.copy()
    if age_encoding == "midpoint":
        result["age"] = result["age"].map(age_mapping)
    elif age_encoding != "onehot":
        raise ValueError("age_encoding must be 'onehot' or 'midpoint'")

    for column in DIAG_COLS:
        result[column] = result[column].map(map_icd9_to_category)
    return result

df_train_clean = apply_feature_engineering(df_train_clean)
df_test_clean = apply_feature_engineering(df_test_clean)

print("Age representation:", AGE_ENCODING)
print("ICD-9 codes mapped to 10 categories, including Supplemental and Other.")
display(df_train_clean["diag_1"].value_counts(dropna=False).to_frame("train_count"))


## 4.5 Target Encoding and Final Feature Matrices


In [ ]:
print("=" * 60)
print("TARGET ENCODING AND GROUP EXTRACTION")
print("=" * 60)

target_mapping = {"NO": 0, ">30": 1, "<30": 2}
target_names = {value: key for key, value in target_mapping.items()}

groups_train = df_train_clean.pop("patient_nbr")
groups_test = df_test_clean.pop("patient_nbr")

y_train = df_train_clean.pop(TARGET).map(target_mapping)
y_test = df_test_clean.pop(TARGET).map(target_mapping)
X_train = df_train_clean.copy()
X_test = df_test_clean.copy()

assert y_train.notna().all() and y_test.notna().all()
assert set(groups_train).isdisjoint(set(groups_test))

print("Target labels are class identifiers: NO=0, >30=1, <30=2")
print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

print("=" * 60)
print("TRAIN-ONLY FEATURE FILTER AND GROUPED CROSS-VALIDATION")
print("=" * 60)

dominance = {
    column: X_train[column].value_counts(dropna=False).iloc[0] / len(X_train)
    for column in X_train.columns
}
constant_train_cols = [
    column for column in X_train.columns
    if X_train[column].nunique(dropna=False) <= 1
]
near_constant_train_cols = [
    column for column, ratio in dominance.items()
    if ratio > 0.95 and column not in constant_train_cols
]

# Default: remove only true constants. Near-constant features remain available
# for an explicit ablation experiment instead of being assumed useless.
DROP_NEAR_CONSTANT = False
columns_to_remove = constant_train_cols + (
    near_constant_train_cols if DROP_NEAR_CONSTANT else []
)

X_train = X_train.drop(columns=columns_to_remove)
X_test = X_test.drop(columns=columns_to_remove)

group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Dropped zero-variance columns:", constant_train_cols)
print("Retained near-constant candidates:", near_constant_train_cols)
print("Group CV:", group_cv)
print("Final raw feature shapes:", X_train.shape, X_test.shape)


## 4.6 Configurable Preprocessing Pipeline


In [ ]:
print("=" * 60)
print("DEFINE FEATURE GROUPS")
print("=" * 60)

numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

assert set(numerical_features).isdisjoint(categorical_features)
assert set(numerical_features + categorical_features) == set(X_train.columns)

print(f"Numerical features ({len(numerical_features)}):")
print(numerical_features)
print(f"Categorical features ({len(categorical_features)}):")
print(categorical_features)


In [ ]:
from modules.preprocessing import get_preprocessor

print("=" * 60)
print("BUILD CONFIGURABLE PREPROCESSOR")
print("=" * 60)

print("Available scaler options: standard, robust, minmax, none")
print("Rare categories are grouped using training-set frequency only.")


In [ ]:
print("=" * 60)
print("VALIDATE THE PREPROCESSOR")
print("=" * 60)

test_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)

X_train_processed = test_preprocessor.fit_transform(X_train)
X_test_processed = test_preprocessor.transform(X_test)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

print("Original train shape:", X_train.shape)
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("No non-finite values remain after preprocessing.")


# Phase 5: Feature Extraction and Dimensionality Reduction

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

print("=" * 60)
print("PCA EXPLAINED VARIANCE - TRAINING DATA ONLY")
print("=" * 60)

preprocessor_pca = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
X_train_pca_ready = preprocessor_pca.fit_transform(X_train)
X_test_pca_ready = preprocessor_pca.transform(X_test)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train_pca_ready)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
threshold_90 = np.argmax(cumulative_variance >= 0.90) + 1
threshold_95 = np.argmax(cumulative_variance >= 0.95) + 1


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.90, color="red", linestyle="--", label=f"90% ({threshold_90} components)")
plt.axhline(0.95, color="green", linestyle="--", label=f"95% ({threshold_95} components)")
plt.axvline(threshold_90, color="red", linestyle="--")
plt.axvline(threshold_95, color="green", linestyle="--")
plt.title("PCA Cumulative Explained Variance - Training Set")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Encoded feature count:", X_train_pca_ready.shape[1])
print("Components for 90% variance:", threshold_90)
print("Components for 95% variance:", threshold_95)

def get_feature_pipeline(pca_variance=None, **preprocessor_options):
    """Create a comparable feature pipeline with optional train-fitted PCA."""
    steps = [("preprocessor", get_preprocessor(
        numerical_features=numerical_features,
        categorical_features=categorical_features,
        **preprocessor_options,
    ))]
    if pca_variance is not None:
        if pca_variance not in (0.90, 0.95):
            raise ValueError("pca_variance must be None, 0.90, or 0.95")
        steps.append(("pca", PCA(n_components=pca_variance, svd_solver="full")))
    return Pipeline(steps)

feature_pipeline_no_pca = get_feature_pipeline(pca_variance=None)
feature_pipeline_pca_90 = get_feature_pipeline(pca_variance=0.90)
feature_pipeline_pca_95 = get_feature_pipeline(pca_variance=0.95)

print("Feature pipelines prepared for downstream grouped model comparison.")


## Export Extracted Feature Matrices


In [ ]:
import json as _json

print("=" * 60)
print("PHASE 5.1: EXPORT EXTRACTED FEATURE MATRICES (.npy)")
print("=" * 60)

# Ensure FEATURES_DIR exists
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Transform full train and test sets using the fitted baseline preprocessor
#    (StandardScaler + OneHotEncoder with min_category_frequency=20)
#    Re-use same configuration as the PCA analysis above.
preprocessor_export = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
X_train_export = preprocessor_export.fit_transform(X_train).astype(np.float32)
X_test_export = preprocessor_export.transform(X_test).astype(np.float32)

# Extract column names after one-hot encoding
feature_names_out = list(preprocessor_export.get_feature_names_out())

# 2. PCA projections (90% and 95% variance)
pca_90_export = PCA(n_components=0.90, svd_solver="full", random_state=RANDOM_STATE)
X_train_pca90 = pca_90_export.fit_transform(X_train_export).astype(np.float32)
X_test_pca90 = pca_90_export.transform(X_test_export).astype(np.float32)

pca_95_export = PCA(n_components=0.95, svd_solver="full", random_state=RANDOM_STATE)
X_train_pca95 = pca_95_export.fit_transform(X_train_export).astype(np.float32)
X_test_pca95 = pca_95_export.transform(X_test_export).astype(np.float32)

y_train_arr = np.asarray(y_train, dtype=np.int64)
y_test_arr = np.asarray(y_test, dtype=np.int64)

# 3. Save all artifacts to FEATURES_DIR
feature_artifacts = {
    "X_train_processed.npy": X_train_export,
    "X_test_processed.npy": X_test_export,
    "X_train_pca90.npy": X_train_pca90,
    "X_test_pca90.npy": X_test_pca90,
    "X_train_pca95.npy": X_train_pca95,
    "X_test_pca95.npy": X_test_pca95,
    "y_train.npy": y_train_arr,
    "y_test.npy": y_test_arr,
    "groups_train.npy": np.asarray(groups_train, dtype=np.int64),
    "groups_test.npy": np.asarray(groups_test, dtype=np.int64),
}

for fname, arr in feature_artifacts.items():
    fpath = FEATURES_DIR / fname
    np.save(fpath, arr)
    print(f"  Saved {fname:30s} shape={str(arr.shape):15s} dtype={arr.dtype}")

# 4. Save feature names for interpretability
feature_names_path = FEATURES_DIR / "feature_names.json"
with open(feature_names_path, "w", encoding="utf-8") as f:
    _json.dump(feature_names_out, f, ensure_ascii=False, indent=2)
print(f"  Saved feature_names.json ({len(feature_names_out)} features)")

print()
print("All feature matrices exported to:", FEATURES_DIR)


# Phase 6: Model Training and Evaluation

## 6.1 Support Vector Machine

Select the Linear SVM configuration by five-fold patient-grouped CV using Macro-F1.


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.dummy import DummyClassifier

from modules.evaluation import evaluate_classifier
from modules.svm_model import (
    build_linear_svm_pipeline,
    run_svm_cross_validation,
)

print("=" * 60)
print("SVM EXPERIMENT SETUP")
print("=" * 60)

LABELS = [0, 1, 2]
CLASS_NAMES = [target_names[label] for label in LABELS]

print("Class mapping:")
for label, name in zip(LABELS, CLASS_NAMES):
    print(f"  {label}: {name}")

print("\nPrimary selection metric: Macro-F1")
print("Cross-validation:", group_cv)

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

print("Evaluation metrics:")
for metric in scoring:
    print("-", metric)

In [ ]:
print("=" * 60)
print("DUMMY CLASSIFIER BASELINE")
print("=" * 60)

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_cv = cross_validate(
    estimator=dummy_model,
    X=X_train,
    y=y_train,
    groups=groups_train,
    cv=group_cv,
    scoring=scoring,
    n_jobs=2,
)

dummy_summary = {
    metric: dummy_cv[f"test_{metric}"].mean()
    for metric in scoring
}

display(
    pd.DataFrame(
        [dummy_summary],
        index=["Dummy - Most Frequent"]
    ).round(4)
)

In [ ]:
print("SVM model and evaluation utilities imported successfully.")

In [ ]:
svm_configs = [
    {
        "name": "SVM-1 | Standard | No PCA | C=1 | No class weight",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": None,
    },
    {
        "name": "SVM-2 | Standard | No PCA | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-3 | MinMax | No PCA | C=1 | Balanced",
        "scaler_type": "minmax",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-4 | Standard | PCA 90% | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.90,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-5 | Standard | PCA 95% | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.95,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-6 | Standard | No PCA | C=0.1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 0.1,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-7 | Standard | No PCA | C=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 10.0,
        "class_weight": "balanced",
    },
]

print(f"Number of SVM configurations: {len(svm_configs)}")

In [ ]:
print("=" * 60)
print("RUN SVM CROSS-VALIDATION EXPERIMENTS")
print("=" * 60)

svm_results = run_svm_cross_validation(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    group_cv=group_cv,
    configs=svm_configs,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scoring=scoring,
    random_state=RANDOM_STATE,
    n_jobs=2,
)

display(svm_results.round(4))

In [ ]:
svm_ranking = svm_results[
    [
        "configuration",
        "cv_accuracy_mean",
        "cv_balanced_accuracy_mean",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_f1_weighted_mean",
    ]
].copy()

display(svm_ranking.round(4))

plot_df = svm_results.sort_values(
    "cv_f1_macro_mean",
    ascending=True
)

ax = plot_df.plot(
    x="configuration",
    y="cv_f1_macro_mean",
    kind="barh",
    figsize=(10, 6),
    legend=False,
)

ax.set_xlabel("Mean Macro-F1")
ax.set_ylabel("Configuration")
ax.set_title("SVM Configuration Comparison - 5-Fold Grouped CV")

plt.tight_layout()
plt.show()

In [ ]:
best_config_name = svm_results.iloc[0]["configuration"]

config_lookup = {
    config["name"]: config
    for config in svm_configs
}

best_config = config_lookup[best_config_name]

print("=" * 60)
print("BEST SVM CONFIGURATION")
print("=" * 60)

print("Configuration:", best_config_name)
print("Scaler:", best_config["scaler_type"])
print("PCA:", best_config["pca_variance"])
print("C:", best_config["C"])
print("Class weight:", best_config["class_weight"])

print(
    "Mean CV Macro-F1:",
    round(svm_results.iloc[0]["cv_f1_macro_mean"], 4)
)

print(
    "Std CV Macro-F1:",
    round(svm_results.iloc[0]["cv_f1_macro_std"], 4)
)

In [ ]:
print("=" * 60)
print("FIT BEST SVM ON FULL TRAINING SET")
print("=" * 60)

best_svm = build_linear_svm_pipeline(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type=best_config["scaler_type"],
    pca_variance=best_config["pca_variance"],
    C=best_config["C"],
    class_weight=best_config["class_weight"],
    random_state=RANDOM_STATE,
)

best_svm.fit(
    X_train,
    y_train,
)

print("Best SVM fitted successfully.")

In [ ]:
print("=" * 60)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 60)

y_test_pred = best_svm.predict(X_test)

In [ ]:
test_metrics, svm_test_summary, report_df = evaluate_classifier(
    y_true=y_test,
    y_pred=y_test_pred,
    labels=LABELS,
    class_names=CLASS_NAMES,
)

test_accuracy = test_metrics["Accuracy"]
test_balanced_accuracy = test_metrics["Balanced Accuracy"]
test_precision_macro = test_metrics["Macro Precision"]
test_recall_macro = test_metrics["Macro Recall"]
test_f1_macro = test_metrics["Macro F1"]
test_f1_weighted = test_metrics["Weighted F1"]

display(svm_test_summary.round(4))

In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

display(report_df.round(4))

In [ ]:
lt30_precision, lt30_recall, lt30_f1, lt30_support = (
    precision_recall_fscore_support(
        y_test,
        y_test_pred,
        labels=[2],
        zero_division=0,
    )
)

lt30_summary = pd.DataFrame(
    {
        "Metric": [
            "Precision",
            "Recall",
            "F1-score",
            "Support",
        ],
        "Value": [
            lt30_precision[0],
            lt30_recall[0],
            lt30_f1[0],
            lt30_support[0],
        ],
    }
)

display(lt30_summary.round(4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    labels=LABELS,
    display_labels=CLASS_NAMES,
    normalize="true",
)

plt.title("Normalized SVM Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ONE-VS-REST AVERAGE PRECISION")
print("=" * 60)

decision_scores = best_svm.decision_function(X_test)

y_test_binary = label_binarize(
    y_test,
    classes=LABELS,
)

macro_average_precision = average_precision_score(
    y_test_binary,
    decision_scores,
    average="macro",
)

lt30_average_precision = average_precision_score(
    (y_test == 2).astype(int),
    decision_scores[:, 2],
)

print(
    "Macro Average Precision (One-vs-Rest):",
    round(macro_average_precision, 4),
)

print(
    "<30 Average Precision (One-vs-Rest):",
    round(lt30_average_precision, 4),
)

In [ ]:
svm_final_result = pd.DataFrame(
    [
        {
            "Model": "Linear SVM",
            "Best Configuration": best_config_name,
            "Accuracy": test_accuracy,
            "Balanced Accuracy": test_balanced_accuracy,
            "Macro Precision": test_precision_macro,
            "Macro Recall": test_recall_macro,
            "Macro F1": test_f1_macro,
            "Weighted F1": test_f1_weighted,
            "<30 Precision": lt30_precision[0],
            "<30 Recall": lt30_recall[0],
            "<30 F1": lt30_f1[0],
            "Macro Average Precision OvR": macro_average_precision,
        }
    ]
)

display(svm_final_result.round(4))

In [ ]:
print("=" * 60)
print("EXPORT FINAL SVM FEATURES")
print("=" * 60)

feature_transformer = best_svm[:-1]
X_train_svm_features = np.asarray(
    feature_transformer.transform(X_train),
    dtype=np.float32,
)
X_test_svm_features = np.asarray(
    feature_transformer.transform(X_test),
    dtype=np.float32,
)

if "pca" in best_svm.named_steps:
    svm_feature_names = np.asarray(
        [
            f"PC{index:03d}"
            for index in range(1, X_train_svm_features.shape[1] + 1)
        ],
        dtype=str,
    )
else:
    svm_feature_names = np.asarray(
        best_svm.named_steps["preprocessor"].get_feature_names_out(),
        dtype=str,
    )

assert X_train_svm_features.shape[0] == len(y_train)
assert X_test_svm_features.shape[0] == len(y_test)
assert X_train_svm_features.shape[1] == X_test_svm_features.shape[1]
assert X_train_svm_features.shape[1] == len(svm_feature_names)
assert X_train_svm_features.dtype == np.float32
assert X_test_svm_features.dtype == np.float32

feature_artifacts = {
    "X_train_svm_features.npy": X_train_svm_features,
    "X_test_svm_features.npy": X_test_svm_features,
    "y_train.npy": np.asarray(y_train, dtype=np.int64),
    "y_test.npy": np.asarray(y_test, dtype=np.int64),
    "svm_feature_names.npy": svm_feature_names,
}

manifest_rows = []
for artifact_name, artifact_data in feature_artifacts.items():
    artifact_path = FEATURES_DIR / artifact_name
    np.save(artifact_path, artifact_data)
    manifest_rows.append(
        {
            "artifact": artifact_name,
            "shape": " x ".join(map(str, artifact_data.shape)),
            "dtype": str(artifact_data.dtype),
            "size_mb": artifact_path.stat().st_size / (1024 ** 2),
            "feature_count": X_train_svm_features.shape[1],
            "best_configuration": best_config_name,
        }
    )

svm_feature_manifest = pd.DataFrame(manifest_rows)
display(svm_feature_manifest.round({"size_mb": 2}))

SVM_REPORT_DIR = REPORTS_DIR / "svm"
SVM_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

svm_results.to_csv(
    SVM_REPORT_DIR / "svm_cv_results.csv",
    index=False,
)

svm_test_summary.to_csv(
    SVM_REPORT_DIR / "svm_test_metrics.csv",
    index=False,
)

svm_final_result.to_csv(
    SVM_REPORT_DIR / "svm_final_summary.csv",
    index=False,
)

report_df.to_csv(
    SVM_REPORT_DIR / "svm_classification_report.csv"
)

svm_feature_manifest.to_csv(
    SVM_REPORT_DIR / "svm_feature_manifest.csv",
    index=False,
)

print("Extracted feature files saved to:")
print(FEATURES_DIR)
print("SVM experiment results saved to:")
print(SVM_REPORT_DIR)

In [ ]:
import joblib

MODEL_PATH = MODELS_DIR / "best_linear_svm.joblib"

joblib.dump(
    best_svm,
    MODEL_PATH,
)

print("Best SVM pipeline saved to:")
print(MODEL_PATH)

## 6.2 Logistic Regression

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import joblib

from modules.logistic_regression_model import (
    train_logistic_regression_pipeline,
)

print("=" * 60)
print("LOGISTIC REGRESSION EXPERIMENT SETUP")
print("=" * 60)

lr_X_train = X_train
lr_X_test = X_test
lr_y_train = y_train
lr_y_test = y_test
LR_EARLY_READMISSION_LABEL = 2

print("Shared train shapes:", lr_X_train.shape, lr_y_train.shape)
print("Shared test shapes:", lr_X_test.shape, lr_y_test.shape)
print("Numerical feature count:", len(numerical_features))
print("Categorical feature count:", len(categorical_features))
print("Early-readmission label:", LR_EARLY_READMISSION_LABEL)
display(lr_y_train.value_counts().sort_index().to_frame("Count"))

### 6.2.1 Data Transformation and PCA Visualization


In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION PCA VISUALIZATION")
print("=" * 60)

lr_visualization_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
lr_X_train_transformed = lr_visualization_preprocessor.fit_transform(
    lr_X_train
)
lr_X_test_transformed = lr_visualization_preprocessor.transform(
    lr_X_test
)

lr_pca_visualization = PCA(
    n_components=2,
    random_state=RANDOM_STATE,
)
lr_X_train_pca = lr_pca_visualization.fit_transform(
    lr_X_train_transformed
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=lr_X_train_pca[:, 0],
    y=lr_X_train_pca[:, 1],
    hue=lr_y_train.map(target_names),
    palette="viridis",
    alpha=0.5,
    s=15,
)
plt.title("2D PCA Visualization of Processed Training Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Readmitted Class")
plt.tight_layout()
plt.show()

del lr_X_train_transformed, lr_X_test_transformed, lr_X_train_pca

### 6.2.2 Warm-start Logistic Regression Training


In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION: OPTIMIZATION PROCESS")
print("=" * 60)
print("Solver: SAGA | Loss: Cross-Entropy | Max Iterations: 100")

(
    lr_pipeline,
    lr_loss_history,
    lr_training_seconds,
) = train_logistic_regression_pipeline(
    X_train=lr_X_train,
    y_train=lr_y_train,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    epochs=100,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    min_category_frequency=20,
)

for row in lr_loss_history.itertuples(index=False):
    print(
        f"Iteration {row.iteration:4d} | "
        f"Cost (Log Loss): {row.log_loss:.4f}"
    )

print("-" * 60)
print(
    "Model training completed successfully in "
    f"{lr_training_seconds:.2f} seconds!"
)

lr_model = lr_pipeline.named_steps["model"]
LR_MODEL_PATH = MODELS_DIR / "logistic_regression_pipeline.joblib"
joblib.dump(lr_pipeline, LR_MODEL_PATH)

print("Full Logistic Regression pipeline saved to:")
print(LR_MODEL_PATH)

### 6.2.3 Sigmoid Decision Boundary Visualization


In [ ]:
print("=" * 60)
print("SIGMOID DECISION BOUNDARY VISUALIZATION")
print("=" * 60)

lr_test_transformed = lr_pipeline.named_steps[
    "preprocessor"
].transform(lr_X_test)
lr_class_index = list(lr_model.classes_).index(
    LR_EARLY_READMISSION_LABEL
)

lr_decision_scores = lr_model.decision_function(
    lr_test_transformed
)
lr_early_scores = (
    lr_decision_scores[:, lr_class_index]
    if lr_decision_scores.ndim > 1
    else lr_decision_scores
)
lr_early_probabilities = lr_model.predict_proba(
    lr_test_transformed
)[:, lr_class_index]

lr_sorted_index = np.argsort(lr_early_scores)
lr_y_test_binary = (
    lr_y_test == LR_EARLY_READMISSION_LABEL
).astype(int)

plt.figure(figsize=(10, 6))
plt.plot(
    lr_early_scores[lr_sorted_index],
    lr_early_probabilities[lr_sorted_index],
    color="red",
    linewidth=3,
    label="Sigmoid Curve (Predicted Probability)",
)
plt.scatter(
    lr_early_scores,
    lr_y_test_binary,
    alpha=0.1,
    color="blue",
    label="Actual Data ('<30' = 1, Other = 0)",
    s=15,
)
plt.title(
    "Logistic Regression: Sigmoid Function for '<30' Readmission Class",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Linear Score (z = W^T X)", fontsize=12)
plt.ylabel("Probability", fontsize=12)
plt.axhline(
    0.5,
    color="gray",
    linestyle="--",
    label="Decision Threshold (0.5)",
)
plt.legend(loc="center right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

del lr_test_transformed

### 6.2.4 Logistic Regression Evaluation


In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION TEST PERFORMANCE")
print("=" * 60)

lr_y_pred = lr_pipeline.predict(lr_X_test)
lr_y_pred_proba = lr_pipeline.predict_proba(lr_X_test)

lr_accuracy = accuracy_score(
    lr_y_test,
    lr_y_pred,
)
lr_roc_auc = roc_auc_score(
    lr_y_test,
    lr_y_pred_proba,
    labels=LABELS,
    multi_class="ovr",
)

print(f"Overall Accuracy : {lr_accuracy:.4f}")
print(f"ROC AUC (OvR)    : {lr_roc_auc:.4f}")
print("\nDetailed Classification Report:")

lr_report_dict = classification_report(
    lr_y_test,
    lr_y_pred,
    labels=LABELS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
print(
    classification_report(
        lr_y_test,
        lr_y_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

lr_report_df = pd.DataFrame(lr_report_dict).T
LR_REPORT_PATH = REPORTS_DIR / "logistic_regression_report.csv"
lr_report_df.to_csv(LR_REPORT_PATH, index=True)

print("Classification report exported to:")
print(LR_REPORT_PATH)

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION CONFUSION MATRIX")
print("=" * 60)

lr_confusion_matrix = confusion_matrix(
    lr_y_test,
    lr_y_pred,
    labels=LABELS,
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    lr_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title(
    "Confusion Matrix - Logistic Regression",
    fontsize=14,
    fontweight="bold",
)
plt.ylabel("Actual Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("TOP 15 LOGISTIC FEATURES FOR '<30' READMISSION")
print("=" * 60)

lr_feature_names = lr_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()
lr_class_index = list(lr_model.classes_).index(
    LR_EARLY_READMISSION_LABEL
)
lr_coefficients = lr_model.coef_[lr_class_index]

lr_coefficient_df = pd.DataFrame(
    {
        "Feature": lr_feature_names,
        "Coefficient": lr_coefficients,
    }
)
lr_coefficient_df["Abs_Coefficient"] = lr_coefficient_df[
    "Coefficient"
].abs()
lr_top_features = (
    lr_coefficient_df
    .sort_values(by="Abs_Coefficient", ascending=False)
    .head(15)
)

display(lr_top_features.round(4))

plt.figure(figsize=(12, 8))
sns.barplot(
    data=lr_top_features,
    x="Coefficient",
    y="Feature",
    hue="Feature",
    palette="coolwarm",
    legend=False,
)
plt.title(
    "Top 15 Most Impactful Features for '<30' Readmission Risk",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel(
    "Logistic Regression Coefficient (Impact on Log-Odds)",
    fontsize=12,
)
plt.ylabel("Patient Feature", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

### 6.2.5 Configuration Comparison with Five-Fold Grouped CV


In [ ]:
print("=" * 60)
print("RUN LOGISTIC REGRESSION CROSS-VALIDATION EXPERIMENTS")
print("=" * 60)

import importlib
import modules.logistic_regression_model
importlib.reload(modules.logistic_regression_model)

try:
    from modules.logistic_regression_model import run_logistic_regression_cross_validation
except ImportError:
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_validate
    from sklearn.pipeline import Pipeline
    from sklearn.exceptions import ConvergenceWarning
    import warnings

    def build_logistic_regression_pipeline(
        numerical_features, categorical_features, scaler_type="standard",
        pca_variance=None, C=1.0, class_weight="balanced", random_state=42,
        solver="lbfgs", max_iter=200, min_category_frequency=20,
    ):
        preprocessor = get_preprocessor(
            numerical_features=numerical_features, categorical_features=categorical_features,
            scaler_type=scaler_type, log_numeric=False, min_category_frequency=min_category_frequency,
        )
        steps = [("preprocessor", preprocessor)]
        if pca_variance is not None:
            steps.append(("pca", PCA(n_components=pca_variance, svd_solver="full", random_state=random_state)))
        steps.append(("model", LogisticRegression(C=C, solver=solver, class_weight=class_weight, max_iter=max_iter, random_state=random_state)))
        return Pipeline(steps)

    def run_logistic_regression_cross_validation(
        X_train, y_train, groups_train, group_cv, configs,
        numerical_features, categorical_features, scoring,
        random_state=42, n_jobs=2, min_category_frequency=20,
    ):
        rows = []
        for config in configs:
            model = build_logistic_regression_pipeline(
                numerical_features=numerical_features, categorical_features=categorical_features,
                scaler_type=config["scaler_type"], pca_variance=config["pca_variance"],
                C=config["C"], class_weight=config["class_weight"], random_state=random_state,
                min_category_frequency=min_category_frequency,
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=ConvergenceWarning)
                result = cross_validate(
                    estimator=model, X=X_train, y=y_train, groups=groups_train,
                    cv=group_cv, scoring=scoring, return_train_score=False,
                    n_jobs=n_jobs, error_score="raise",
                )
            rows.append({
                "configuration": config["name"],
                "scaler": config["scaler_type"],
                "pca": "None" if config["pca_variance"] is None else f"{int(config['pca_variance'] * 100)}%",
                "C": config["C"],
                "class_weight": str(config["class_weight"]),
                "cv_accuracy_mean": result["test_accuracy"].mean(),
                "cv_balanced_accuracy_mean": result["test_balanced_accuracy"].mean(),
                "cv_precision_macro_mean": result["test_precision_macro"].mean(),
                "cv_recall_macro_mean": result["test_recall_macro"].mean(),
                "cv_f1_macro_mean": result["test_f1_macro"].mean(),
                "cv_f1_macro_std": result["test_f1_macro"].std(),
                "cv_f1_weighted_mean": result["test_f1_weighted"].mean(),
                "fit_time_mean": result["fit_time"].mean(),
            })
        return pd.DataFrame(rows).sort_values(by="cv_f1_macro_mean", ascending=False).reset_index(drop=True)

lr_configs = [
    {
        "name": "LR-1 | Standard | No PCA | C=1.0 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "LR-2 | Standard | No PCA | C=0.1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 0.1,
        "class_weight": "balanced",
    },
    {
        "name": "LR-3 | Standard | No PCA | C=10.0 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 10.0,
        "class_weight": "balanced",
    },
    {
        "name": "LR-4 | MinMax | No PCA | C=1.0 | Balanced",
        "scaler_type": "minmax",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "LR-5 | Standard | PCA 90% | C=1.0 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.90,
        "C": 1.0,
        "class_weight": "balanced",
    },
]

lr_cv_results = run_logistic_regression_cross_validation(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    group_cv=group_cv,
    configs=lr_configs,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scoring=scoring,
    random_state=RANDOM_STATE,
    n_jobs=2,
    min_category_frequency=20,
)

display(lr_cv_results.round(4))

# Plot CV comparison
plt.figure(figsize=(10, 5))
sns.barplot(
    data=lr_cv_results,
    x="cv_f1_macro_mean",
    y="configuration",
    hue="configuration",
    palette="viridis",
    legend=False,
)
plt.title("Logistic Regression Configuration Comparison (5-Fold Group CV)", fontsize=13, fontweight="bold")
plt.xlabel("Mean Validation Macro-F1", fontsize=11)
plt.ylabel("Configuration", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


### 6.2.6 Final Result and Artifact Export


In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION FINAL RESULT SUMMARY")
print("=" * 60)

from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support, average_precision_score
from sklearn.preprocessing import label_binarize

# Compute standard classification metrics on held-out test set
lr_balanced_acc = balanced_accuracy_score(lr_y_test, lr_y_pred)
lr_prec_macro = lr_report_dict["macro avg"]["precision"]
lr_rec_macro = lr_report_dict["macro avg"]["recall"]
lr_f1_macro = lr_report_dict["macro avg"]["f1-score"]
lr_f1_weighted = lr_report_dict["weighted avg"]["f1-score"]

# Extract class-level metrics for "<30" (label 2)
lr_lt30_prec = lr_report_dict["<30"]["precision"]
lr_lt30_rec = lr_report_dict["<30"]["recall"]
lr_lt30_f1 = lr_report_dict["<30"]["f1-score"]

# Multiclass Average Precision (OvR)
lr_y_test_binary = label_binarize(lr_y_test, classes=LABELS)
lr_macro_avg_precision = average_precision_score(
    lr_y_test_binary,
    lr_y_pred_proba,
    average="macro",
)

lr_best_config_name = "LR-1 | Standard | No PCA | SAGA 100 Epochs | Balanced"

lr_final_result = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Best Configuration": lr_best_config_name,
            "Accuracy": lr_accuracy,
            "Balanced Accuracy": lr_balanced_acc,
            "Macro Precision": lr_prec_macro,
            "Macro Recall": lr_rec_macro,
            "Macro F1": lr_f1_macro,
            "Weighted F1": lr_f1_weighted,
            "<30 Precision": lr_lt30_prec,
            "<30 Recall": lr_lt30_rec,
            "<30 F1": lr_lt30_f1,
            "Macro ROC-AUC OvR": lr_roc_auc,
            "Macro Average Precision OvR": lr_macro_avg_precision,
        }
    ]
)

display(lr_final_result.round(4))

# ── Save LR Reports ──
LR_REPORT_DIR = REPORTS_DIR / "lr"
LR_REPORT_DIR.mkdir(parents=True, exist_ok=True)

lr_final_result.to_csv(LR_REPORT_DIR / "lr_final_summary.csv", index=False)
lr_report_df.to_csv(LR_REPORT_DIR / "lr_classification_report.csv", index=True)
lr_cv_results.to_csv(LR_REPORT_DIR / "lr_cv_results.csv", index=False)

# Test metrics table
lr_test_summary = pd.DataFrame(
    [
        {"Metric": "Accuracy", "Score": lr_accuracy},
        {"Metric": "Balanced Accuracy", "Score": lr_balanced_acc},
        {"Metric": "Macro Precision", "Score": lr_prec_macro},
        {"Metric": "Macro Recall", "Score": lr_rec_macro},
        {"Metric": "Macro F1", "Score": lr_f1_macro},
        {"Metric": "Weighted F1", "Score": lr_f1_weighted},
        {"Metric": "<30 Precision", "Score": lr_lt30_prec},
        {"Metric": "<30 Recall", "Score": lr_lt30_rec},
        {"Metric": "<30 F1", "Score": lr_lt30_f1},
        {"Metric": "Macro ROC-AUC OvR", "Score": lr_roc_auc},
        {"Metric": "Macro Average Precision OvR", "Score": lr_macro_avg_precision},
    ]
)
lr_test_summary.to_csv(LR_REPORT_DIR / "lr_test_metrics.csv", index=False)

print("\nLogistic Regression experiment results saved to:")
print(LR_REPORT_DIR)


## 6.3 Random Forest


In [ ]:
from sklearn.metrics import (
    auc,
    average_precision_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize

from modules.random_forest_model import (
    build_rf_pipeline,
    run_rf_cross_validation,
)

print("=" * 60)
print("RANDOM FOREST EXPERIMENT SETUP")
print("=" * 60)

rf_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

print("Primary selection metric: Macro-F1")
print("Cross-validation:", group_cv)
print("Evaluation metrics:")
for metric in rf_scoring:
    print("-", metric)

In [ ]:
print("=" * 60)
print("RANDOM FOREST DUMMY CLASSIFIER BASELINE")
print("=" * 60)

rf_dummy_model = DummyClassifier(
    strategy="most_frequent"
)
rf_dummy_cv = cross_validate(
    estimator=rf_dummy_model,
    X=X_train,
    y=y_train,
    groups=groups_train,
    cv=group_cv,
    scoring=rf_scoring,
    n_jobs=2,
)
rf_dummy_summary = {
    metric: rf_dummy_cv[f"test_{metric}"].mean()
    for metric in rf_scoring
}

display(
    pd.DataFrame(
        [rf_dummy_summary],
        index=["Dummy - Most Frequent"],
    ).round(4)
)

In [ ]:
rf_configs = [
    {
        "name": "RF-1 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | No class weight",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None,
    },
    {
        "name": "RF-2 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-3 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced Subsample",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
    },
    {
        "name": "RF-4 | MinMax | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "minmax",
        "feature_range": (0, 1),
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-5 | Standard | PCA 90% | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.90,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-6 | Standard | PCA 95% | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.95,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-7 | Standard | No PCA | Trees=100 | Depth=10 | Leaf=20 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 20,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-8 | Standard | No PCA | Trees=100 | Depth=18 | Leaf=5 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 18,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-9 | Standard | No PCA | Trees=200 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 200,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
]

print(
    "Number of Random Forest configurations:",
    len(rf_configs),
)

In [ ]:
print("=" * 60)
print("RUN RANDOM FOREST CROSS-VALIDATION EXPERIMENTS")
print("=" * 60)

rf_results = run_rf_cross_validation(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    group_cv=group_cv,
    configs=rf_configs,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scoring=rf_scoring,
    random_state=RANDOM_STATE,
    cv_n_jobs=1,
    model_n_jobs=-1,
    min_category_frequency=20,
)

display(rf_results.round(4))

In [ ]:
rf_ranking = rf_results[
    [
        "configuration",
        "cv_accuracy_mean",
        "cv_balanced_accuracy_mean",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_f1_weighted_mean",
    ]
].copy()

display(rf_ranking.round(4))

rf_plot_df = rf_results.sort_values(
    "cv_f1_macro_mean",
    ascending=True,
)
rf_axis = rf_plot_df.plot(
    x="configuration",
    y="cv_f1_macro_mean",
    kind="barh",
    figsize=(11, 7),
    legend=False,
    color="#2b5c8f",
)
rf_axis.set_xlabel("Mean Macro-F1", fontsize=11)
rf_axis.set_ylabel("Configuration", fontsize=11)
rf_axis.set_title(
    "Random Forest Configuration Comparison - 5-Fold Grouped CV",
    fontsize=13,
    fontweight="bold",
)
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rf_best_config_name = rf_results.iloc[0][
    "configuration"
]
rf_config_lookup = {
    config["name"]: config
    for config in rf_configs
}
rf_best_config = rf_config_lookup[
    rf_best_config_name
]

print("=" * 60)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 60)
print("Configuration:", rf_best_config_name)
print("Scaler:", rf_best_config["scaler_type"])
print("PCA:", rf_best_config["pca_variance"])
print("Trees:", rf_best_config["n_estimators"])
print("Max Depth:", rf_best_config["max_depth"])
print(
    "Min Samples Leaf:",
    rf_best_config["min_samples_leaf"],
)
print(
    "Class Weight:",
    rf_best_config["class_weight"],
)
print(
    "Mean CV Macro-F1:",
    round(
        rf_results.iloc[0]["cv_f1_macro_mean"],
        4,
    ),
)
print(
    "Std CV Macro-F1:",
    round(
        rf_results.iloc[0]["cv_f1_macro_std"],
        4,
    ),
)

In [ ]:
print("=" * 60)
print("FIT BEST RANDOM FOREST ON FULL TRAINING SET")
print("=" * 60)

rf_best_model = build_rf_pipeline(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type=rf_best_config["scaler_type"],
    feature_range=rf_best_config.get(
        "feature_range",
        (0, 1),
    ),
    pca_variance=rf_best_config["pca_variance"],
    n_estimators=rf_best_config["n_estimators"],
    max_depth=rf_best_config["max_depth"],
    min_samples_leaf=rf_best_config[
        "min_samples_leaf"
    ],
    max_features=rf_best_config["max_features"],
    class_weight=rf_best_config["class_weight"],
    random_state=RANDOM_STATE,
    model_n_jobs=-1,
    min_category_frequency=20,
)
rf_best_model.fit(
    X_train,
    y_train,
)

print(
    "Best Random Forest fitted successfully "
    "on the full training set."
)

In [ ]:
print("=" * 60)
print("RANDOM FOREST HELD-OUT TEST EVALUATION")
print("=" * 60)

rf_y_test_pred = rf_best_model.predict(
    X_test
)
rf_y_test_probability = rf_best_model.predict_proba(
    X_test
)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_y_test_pred,
)
rf_test_balanced_accuracy = balanced_accuracy_score(
    y_test,
    rf_y_test_pred,
)
rf_test_precision_macro = precision_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_recall_macro = recall_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_f1_macro = f1_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_f1_weighted = f1_score(
    y_test,
    rf_y_test_pred,
    average="weighted",
    zero_division=0,
)
rf_test_roc_auc_macro = roc_auc_score(
    y_test,
    rf_y_test_probability,
    labels=LABELS,
    multi_class="ovr",
    average="macro",
)

rf_test_summary = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "Weighted F1",
            "Macro ROC-AUC (OvR)",
        ],
        "Score": [
            rf_test_accuracy,
            rf_test_balanced_accuracy,
            rf_test_precision_macro,
            rf_test_recall_macro,
            rf_test_f1_macro,
            rf_test_f1_weighted,
            rf_test_roc_auc_macro,
        ],
    }
)

display(rf_test_summary.round(4))

In [ ]:
print("=" * 60)
print("RANDOM FOREST CLASSIFICATION REPORT")
print("=" * 60)

rf_report = classification_report(
    y_test,
    rf_y_test_pred,
    labels=LABELS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
rf_report_df = pd.DataFrame(rf_report).T
display(rf_report_df.round(4))

(
    rf_lt30_precision,
    rf_lt30_recall,
    rf_lt30_f1,
    rf_lt30_support,
) = precision_recall_fscore_support(
    y_test,
    rf_y_test_pred,
    labels=[2],
    zero_division=0,
)
rf_lt30_summary = pd.DataFrame(
    {
        "Metric": [
            "Precision",
            "Recall",
            "F1-score",
            "Support",
        ],
        "Value": [
            rf_lt30_precision[0],
            rf_lt30_recall[0],
            rf_lt30_f1[0],
            rf_lt30_support[0],
        ],
    }
)
display(rf_lt30_summary.round(4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_y_test_pred,
    labels=LABELS,
    display_labels=CLASS_NAMES,
    normalize="true",
    cmap="Blues",
)
plt.title(
    "Normalized Random Forest Confusion Matrix - Test Set",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("MULTICLASS ROC & PRECISION-RECALL CURVES (ONE-VS-REST)")
print("=" * 60)

rf_y_test_binary = label_binarize(
    y_test,
    classes=LABELS,
)
rf_macro_average_precision = average_precision_score(
    rf_y_test_binary,
    rf_y_test_probability,
    average="macro",
)
rf_lt30_average_precision = average_precision_score(
    (y_test == 2).astype(int),
    rf_y_test_probability[:, 2],
)

print(
    "Macro Average Precision (One-vs-Rest):",
    round(rf_macro_average_precision, 4),
)
print(
    "<30 Average Precision (One-vs-Rest):",
    round(rf_lt30_average_precision, 4),
)

rf_figure, rf_axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
)
rf_colors = ["#2b5c8f", "#d95f02", "#e7298a"]

for index, (color, class_name) in enumerate(
    zip(rf_colors, CLASS_NAMES)
):
    rf_fpr, rf_tpr, _ = roc_curve(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_class_auc = auc(rf_fpr, rf_tpr)
    rf_axes[0].plot(
        rf_fpr,
        rf_tpr,
        color=color,
        lw=2,
        label=(
            f"Class '{class_name}' "
            f"(AUC = {rf_class_auc:.3f})"
        ),
    )

rf_axes[0].plot(
    [0, 1],
    [0, 1],
    "k--",
    lw=1.2,
    label="Random (AUC = 0.500)",
)
rf_axes[0].set_xlabel("False Positive Rate")
rf_axes[0].set_ylabel("True Positive Rate")
rf_axes[0].set_title(
    f"ROC Curves OvR "
    f"(Macro AUC = {rf_test_roc_auc_macro:.3f})",
    fontweight="bold",
)
rf_axes[0].legend(loc="lower right")
rf_axes[0].grid(True, alpha=0.3)

for index, (color, class_name) in enumerate(
    zip(rf_colors, CLASS_NAMES)
):
    (
        rf_precision_curve,
        rf_recall_curve,
        _,
    ) = precision_recall_curve(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_pr_auc = average_precision_score(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_axes[1].plot(
        rf_recall_curve,
        rf_precision_curve,
        color=color,
        lw=2,
        label=(
            f"Class '{class_name}' "
            f"(AP = {rf_pr_auc:.3f})"
        ),
    )

rf_axes[1].set_xlabel("Recall")
rf_axes[1].set_ylabel("Precision")
rf_axes[1].set_title(
    f"Precision-Recall Curves "
    f"(Macro AP = {rf_macro_average_precision:.3f})",
    fontweight="bold",
)
rf_axes[1].legend(loc="upper right")
rf_axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
rf_final_result = pd.DataFrame(
    [
        {
            "Model": "Random Forest",
            "Best Configuration": (
                rf_best_config_name
            ),
            "Accuracy": rf_test_accuracy,
            "Balanced Accuracy": (
                rf_test_balanced_accuracy
            ),
            "Macro Precision": (
                rf_test_precision_macro
            ),
            "Macro Recall": rf_test_recall_macro,
            "Macro F1": rf_test_f1_macro,
            "Weighted F1": rf_test_f1_weighted,
            "<30 Precision": rf_lt30_precision[0],
            "<30 Recall": rf_lt30_recall[0],
            "<30 F1": rf_lt30_f1[0],
            "Macro ROC-AUC OvR": (
                rf_test_roc_auc_macro
            ),
            "Macro Average Precision OvR": (
                rf_macro_average_precision
            ),
        }
    ]
)

display(rf_final_result.round(4))

In [ ]:
print("=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

rf_preprocessor_fitted = rf_best_model.named_steps[
    "preprocessor"
]
rf_feature_names = np.asarray(
    rf_preprocessor_fitted.get_feature_names_out()
)
rf_importances = rf_best_model.named_steps[
    "rf"
].feature_importances_
rf_sorted_index = np.argsort(
    rf_importances
)[::-1]
RF_TOP_K = 20

rf_top_features_df = pd.DataFrame(
    {
        "Rank": np.arange(1, RF_TOP_K + 1),
        "Feature Name": rf_feature_names[
            rf_sorted_index[:RF_TOP_K]
        ],
        "Importance": rf_importances[
            rf_sorted_index[:RF_TOP_K]
        ],
        "Cumulative (%)": np.cumsum(
            rf_importances[
                rf_sorted_index[:RF_TOP_K]
            ]
        ) * 100,
    }
)

display(rf_top_features_df.round(4))

plt.figure(figsize=(11, 7))
sns.barplot(
    data=rf_top_features_df,
    x="Importance",
    y="Feature Name",
    hue="Feature Name",
    palette="Blues_r",
    legend=False,
)
plt.title(
    f"Top {RF_TOP_K} Most Important Features - Best Random Forest",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel(
    "Gini Feature Importance (Mean Decrease in Impurity)"
)
plt.ylabel("Feature")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("EXPORT RANDOM FOREST RESULTS")
print("=" * 60)

# ── Save RF experiment reports ──
RF_REPORT_DIR = REPORTS_DIR / "rf"
RF_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

rf_results.to_csv(
    RF_REPORT_DIR / "rf_cv_results.csv",
    index=False,
)
rf_test_summary.to_csv(
    RF_REPORT_DIR / "rf_test_metrics.csv",
    index=False,
)
rf_final_result.to_csv(
    RF_REPORT_DIR / "rf_final_summary.csv",
    index=False,
)
rf_report_df.to_csv(
    RF_REPORT_DIR / "rf_classification_report.csv"
)
rf_top_features_df.to_csv(
    RF_REPORT_DIR / "rf_top_features.csv",
    index=False,
)

# ── Save best RF model checkpoint ──
RF_MODEL_PATH = MODELS_DIR / "best_random_forest.joblib"
joblib.dump(
    rf_best_model,
    RF_MODEL_PATH,
)

print("Random Forest experiment results saved to:")
print(RF_REPORT_DIR)
print("Best Random Forest pipeline saved to:")
print(RF_MODEL_PATH)


## 6.4 Model Comparison


In [ ]:
print("=" * 70)
print("MASTER MODEL COMPARISON - HELD-OUT TEST PERFORMANCE")
print("=" * 70)

# 1. Combine final test evaluation summaries from all 3 models
master_comparison_df = pd.concat(
    [svm_final_result, lr_final_result, rf_final_result],
    ignore_index=True,
)

# Reorder columns logically
cols_order = [
    "Model",
    "Best Configuration",
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
    "Weighted F1",
    "Macro Precision",
    "Macro Recall",
    "<30 Precision",
    "<30 Recall",
    "<30 F1",
    "Macro ROC-AUC OvR",
    "Macro Average Precision OvR",
]

# Ensure all columns exist (fill missing e.g. LinearSVM ROC-AUC with NaN)
for col in cols_order:
    if col not in master_comparison_df.columns:
        master_comparison_df[col] = np.nan

master_comparison_df = master_comparison_df[cols_order].sort_values(
    by="Macro F1", ascending=False
).reset_index(drop=True)

# Display Master Table
display(master_comparison_df.round(4))

# 2. Export Master Comparison Table to reports/
master_comparison_path = REPORTS_DIR / "master_model_comparison.csv"
master_comparison_df.to_csv(master_comparison_path, index=False)
print(f"\nMaster model comparison table successfully exported to:")
print(f" -> {master_comparison_path}")


In [ ]:
print("=" * 70)
print("VISUALIZING MODEL PERFORMANCE ACROSS COMPETING OBJECTIVES")
print("=" * 70)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

palette = ["#2b5c8f", "#d95f02", "#2ca02c"]

# ── Subplot 1: Overall Classification Metrics ──
overall_metrics = ["Accuracy", "Balanced Accuracy", "Macro F1", "Weighted F1"]
overall_df = master_comparison_df.melt(
    id_vars="Model",
    value_vars=overall_metrics,
    var_name="Metric",
    value_name="Score",
)

sns.barplot(
    data=overall_df,
    x="Metric",
    y="Score",
    hue="Model",
    palette=palette,
    ax=axes[0],
)
axes[0].set_title("Overall Performance Metrics (Test Set)", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, 0.7)
axes[0].set_ylabel("Score", fontsize=11)
axes[0].set_xlabel("")
axes[0].legend(loc="upper right", frameon=True)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[0].annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h / 2),
                         ha="center", va="center", fontsize=8, color="white", fontweight="bold", rotation=90)

# ── Subplot 2: Critical Early Readmission (<30 Days) Performance ──
minority_metrics = ["<30 Precision", "<30 Recall", "<30 F1"]
minority_df = master_comparison_df.melt(
    id_vars="Model",
    value_vars=minority_metrics,
    var_name="Metric",
    value_name="Score",
)

sns.barplot(
    data=minority_df,
    x="Metric",
    y="Score",
    hue="Model",
    palette=palette,
    ax=axes[1],
)
axes[1].set_title("Minority Class '<30' Readmission Metrics", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 0.7)
axes[1].set_ylabel("Score", fontsize=11)
axes[1].set_xlabel("")
axes[1].legend(loc="upper right", frameon=True)
for p in axes[1].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[1].annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h / 2),
                         ha="center", va="center", fontsize=8, color="white", fontweight="bold", rotation=90)

# ── Subplot 3: Ranking & Discrimination Capability ──
auc_metrics = ["Macro ROC-AUC OvR", "Macro Average Precision OvR"]
auc_df = master_comparison_df.melt(
    id_vars="Model",
    value_vars=auc_metrics,
    var_name="Metric",
    value_name="Score",
)

sns.barplot(
    data=auc_df,
    x="Metric",
    y="Score",
    hue="Model",
    palette=palette,
    ax=axes[2],
)
axes[2].set_title("Discrimination & Area Under Curve (OvR)", fontsize=13, fontweight="bold")
axes[2].set_ylim(0, 0.75)
axes[2].set_ylabel("Score", fontsize=11)
axes[2].set_xlabel("")
axes[2].legend(loc="upper right", frameon=True)
for p in axes[2].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[2].annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h / 2),
                         ha="center", va="center", fontsize=8, color="white", fontweight="bold", rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
print("=" * 70)
print("CROSS-VALIDATION PERFORMANCE ACROSS ALL TESTED CONFIGURATIONS")
print("=" * 70)

cv_list = []

if "svm_results" in locals():
    svm_cv_copy = svm_results.copy()
    svm_cv_copy["Model Family"] = "Linear SVM"
    cv_list.append(svm_cv_copy)

if "lr_cv_results" in locals():
    lr_cv_copy = lr_cv_results.copy()
    lr_cv_copy["Model Family"] = "Logistic Regression"
    cv_list.append(lr_cv_copy)

if "rf_results" in locals():
    rf_cv_copy = rf_results.copy()
    rf_cv_copy["Model Family"] = "Random Forest"
    cv_list.append(rf_cv_copy)

if cv_list:
    shared_cols = [
        "Model Family",
        "configuration",
        "scaler",
        "pca",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_accuracy_mean",
        "cv_balanced_accuracy_mean",
        "fit_time_mean",
    ]

    all_cv_df = pd.concat(cv_list, ignore_index=True)
    all_cv_df = all_cv_df[[c for c in shared_cols if c in all_cv_df.columns]]
    all_cv_df = all_cv_df.sort_values(by="cv_f1_macro_mean", ascending=False).reset_index(drop=True)

    print("TOP 10 CONFIGURATIONS ACROSS ALL TESTED MODELS (Sorted by CV Macro-F1):")
    display(all_cv_df.head(10).round(4))

    # Export configuration comparison CSV
    cv_landscape_path = REPORTS_DIR / "master_cv_configurations_comparison.csv"
    all_cv_df.to_csv(cv_landscape_path, index=False)
    print(f"\nAll CV configurations exported to:\n -> {cv_landscape_path}")

    # Plot Top configurations with error bars
    top_n = min(12, len(all_cv_df))
    top_plot_df = all_cv_df.head(top_n)
    plt.figure(figsize=(14, 7))
    sns.barplot(
        data=top_plot_df,
        x="cv_f1_macro_mean",
        y="configuration",
        hue="Model Family",
        dodge=False,
        palette="tab10",
    )
    plt.errorbar(
        x=top_plot_df["cv_f1_macro_mean"],
        y=range(len(top_plot_df)),
        xerr=top_plot_df["cv_f1_macro_std"],
        fmt="none",
        c="black",
        capsize=3,
        label="CV Standard Deviation (5 folds)",
    )
    plt.title("Top Tested Configurations (Validation Macro-F1 ± Std across 5 Folds)", fontsize=13, fontweight="bold")
    plt.xlabel("Mean 5-Fold Grouped CV Macro-F1", fontsize=11)
    plt.ylabel("Configuration Name", fontsize=11)
    plt.legend(loc="lower right")
    plt.grid(axis="x", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


# Phase 7: MLP Training and Evaluation

Validation preprocessing is fitted on sub-training data; final preprocessing is fitted on the complete outer training set.


## 7.1 MLP Validation Split and Preprocessing


In [ ]:
import os
import time
import importlib

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader

import modules.mlp_model
importlib.reload(modules.mlp_model)
from modules.mlp_model import (
    CLASS_NAMES,
    LABELS,
    DiabetesMLP,
    DiabetesTabularDataset,
    evaluate_mlp,
    train_mlp_fixed_epochs,
    train_mlp_pipeline,
)

print("=" * 70)
print("PHASE 7.1: MLP VALIDATION SPLIT AND PREPROCESSING")
print("=" * 70)

groups_train_arr = np.asarray(groups_train, dtype=np.int64)
groups_test_arr = np.asarray(groups_test, dtype=np.int64)
y_mlp_outer_train = np.asarray(y_train, dtype=np.int64)
y_mlp_outer_test = np.asarray(y_test, dtype=np.int64)

assert len(X_train) == len(y_mlp_outer_train) == len(groups_train_arr)
assert len(X_test) == len(y_mlp_outer_test) == len(groups_test_arr)
assert set(groups_train_arr).isdisjoint(set(groups_test_arr))

# Use one stratified patient-group fold as the inner validation holdout.
mlp_validation_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
subtrain_idx, validation_idx = next(
    mlp_validation_cv.split(
        X_train,
        y_mlp_outer_train,
        groups=groups_train_arr,
    )
)

subtrain_groups = set(groups_train_arr[subtrain_idx])
validation_groups = set(groups_train_arr[validation_idx])
assert subtrain_groups.isdisjoint(validation_groups)
assert set(np.unique(y_mlp_outer_train[subtrain_idx])) == set(LABELS)
assert set(np.unique(y_mlp_outer_train[validation_idx])) == set(LABELS)

# Fit preprocessing only on the inner sub-training partition.
mlp_selection_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
X_mlp_subtrain = mlp_selection_preprocessor.fit_transform(
    X_train.iloc[subtrain_idx]
).astype(np.float32)
X_mlp_validation = mlp_selection_preprocessor.transform(
    X_train.iloc[validation_idx]
).astype(np.float32)

assert X_mlp_subtrain.shape[1] == X_mlp_validation.shape[1]
assert np.isfinite(X_mlp_subtrain).all()
assert np.isfinite(X_mlp_validation).all()


def compute_balanced_class_weights(labels):
    counts = np.bincount(labels, minlength=len(LABELS))
    if np.any(counts == 0):
        raise ValueError("Every training partition must contain all target classes.")
    return counts, len(labels) / (len(LABELS) * counts)


subtrain_class_counts, selection_class_weights = compute_balanced_class_weights(
    y_mlp_outer_train[subtrain_idx]
)

mlp_subtrain_dataset = DiabetesTabularDataset(
    X_mlp_subtrain,
    y_mlp_outer_train[subtrain_idx],
)
mlp_validation_dataset = DiabetesTabularDataset(
    X_mlp_validation,
    y_mlp_outer_train[validation_idx],
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
MLP_BATCH_SIZE = 512
MLP_LEARNING_RATE = 1e-3
MLP_WEIGHT_DECAY = 1e-4

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
selection_generator = torch.Generator().manual_seed(RANDOM_STATE)

mlp_subtrain_loader = DataLoader(
    mlp_subtrain_dataset,
    batch_size=MLP_BATCH_SIZE,
    shuffle=True,
    generator=selection_generator,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=(len(mlp_subtrain_dataset) % MLP_BATCH_SIZE == 1),
)
mlp_validation_loader = DataLoader(
    mlp_validation_dataset,
    batch_size=MLP_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

mlp_selection_model = DiabetesMLP(input_dim=X_mlp_subtrain.shape[1])
MLP_REPORT_DIR = REPORTS_DIR / "mlp"
MLP_REPORT_DIR.mkdir(parents=True, exist_ok=True)
MLP_CHECKPOINT_PATH = MODELS_DIR / "best_mlp.pt"
MLP_PREPROCESSOR_PATH = MODELS_DIR / "mlp_preprocessor.joblib"

print(f"Compute device: {device}")
print(
    "Selection partition sizes -> "
    f"Sub-train: {len(subtrain_idx):,} | "
    f"Validation: {len(validation_idx):,}"
)
print(
    "Sub-train class frequencies [NO, >30, <30]:",
    subtrain_class_counts.tolist(),
)
print(
    "Selection class weights:",
    np.round(selection_class_weights, 4).tolist(),
)
print(f"Selection feature count: {X_mlp_subtrain.shape[1]}")


## 7.2 MLP Epoch Selection and Full-Train Refit


In [ ]:
print("=" * 70)
print("PHASE 7.2: MLP EPOCH SELECTION AND FULL-TRAIN REFIT")
print("=" * 70)

# Stage 1: select the epoch count using leakage-safe inner validation.
selection_started = time.perf_counter()
(
    mlp_selection_model,
    mlp_selection_history,
    mlp_best_epoch,
    mlp_best_val_f1,
) = train_mlp_pipeline(
    model=mlp_selection_model,
    train_loader=mlp_subtrain_loader,
    validation_loader=mlp_validation_loader,
    class_weights=selection_class_weights,
    device=device,
    checkpoint_path=None,
    learning_rate=MLP_LEARNING_RATE,
    weight_decay=MLP_WEIGHT_DECAY,
    epochs=100,
    early_stopping_patience=12,
    scheduler_patience=5,
)
mlp_selection_seconds = time.perf_counter() - selection_started
mlp_selection_history.to_csv(
    MLP_REPORT_DIR / "mlp_training_history.csv",
    index=False,
)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(
    mlp_selection_history["epoch"],
    mlp_selection_history["train_loss"],
    label="Train Loss",
)
axes[0].plot(
    mlp_selection_history["epoch"],
    mlp_selection_history["validation_loss"],
    label="Validation Loss",
)
axes[0].axvline(
    mlp_best_epoch,
    color="gray",
    linestyle="--",
    label=f"Best Epoch ({mlp_best_epoch})",
)
axes[0].set_title("MLP Selection Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Weighted Cross-Entropy")
axes[0].legend()

axes[1].plot(
    mlp_selection_history["epoch"],
    mlp_selection_history["train_macro_f1"],
    label="Train Macro-F1",
)
axes[1].plot(
    mlp_selection_history["epoch"],
    mlp_selection_history["validation_macro_f1"],
    label="Validation Macro-F1",
)
axes[1].axvline(
    mlp_best_epoch,
    color="gray",
    linestyle="--",
    label=f"Best Epoch ({mlp_best_epoch})",
)
axes[1].set_title("MLP Selection Macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
axes[1].legend()
fig.tight_layout()
fig.savefig(
    MLP_REPORT_DIR / "mlp_training_curves.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Replay the learning rate that was actually used at each selected epoch.
selection_learning_rates = mlp_selection_history["learning_rate"].to_numpy(
    dtype=float
)[:mlp_best_epoch]
mlp_refit_learning_rates = np.concatenate(
    ([MLP_LEARNING_RATE], selection_learning_rates[:-1])
)
assert len(mlp_refit_learning_rates) == mlp_best_epoch

# Stage 2: refit preprocessing and a fresh MLP on the complete outer train set.
mlp_final_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
X_mlp_final_train = mlp_final_preprocessor.fit_transform(X_train).astype(
    np.float32
)
X_mlp_final_test = mlp_final_preprocessor.transform(X_test).astype(np.float32)

assert X_mlp_final_train.shape[1] == X_mlp_final_test.shape[1]
assert np.isfinite(X_mlp_final_train).all()
assert np.isfinite(X_mlp_final_test).all()
joblib.dump(mlp_final_preprocessor, MLP_PREPROCESSOR_PATH)

full_train_class_counts, final_class_weights = compute_balanced_class_weights(
    y_mlp_outer_train
)
mlp_final_train_dataset = DiabetesTabularDataset(
    X_mlp_final_train,
    y_mlp_outer_train,
)
mlp_test_dataset = DiabetesTabularDataset(
    X_mlp_final_test,
    y_mlp_outer_test,
)

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
refit_generator = torch.Generator().manual_seed(RANDOM_STATE)

mlp_final_train_loader = DataLoader(
    mlp_final_train_dataset,
    batch_size=MLP_BATCH_SIZE,
    shuffle=True,
    generator=refit_generator,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=(len(mlp_final_train_dataset) % MLP_BATCH_SIZE == 1),
)
mlp_test_loader = DataLoader(
    mlp_test_dataset,
    batch_size=MLP_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

mlp_model = DiabetesMLP(input_dim=X_mlp_final_train.shape[1])
refit_started = time.perf_counter()
mlp_model, mlp_final_refit_history = train_mlp_fixed_epochs(
    model=mlp_model,
    train_loader=mlp_final_train_loader,
    class_weights=final_class_weights,
    device=device,
    epochs=mlp_best_epoch,
    learning_rate=MLP_LEARNING_RATE,
    learning_rates=mlp_refit_learning_rates,
    weight_decay=MLP_WEIGHT_DECAY,
    checkpoint_path=MLP_CHECKPOINT_PATH,
)
mlp_refit_seconds = time.perf_counter() - refit_started
mlp_training_seconds = mlp_selection_seconds + mlp_refit_seconds
mlp_final_refit_history.to_csv(
    MLP_REPORT_DIR / "mlp_final_refit_history.csv",
    index=False,
)

assert len(mlp_final_refit_history) == mlp_best_epoch
print(f"Best selection epoch: {mlp_best_epoch}")
print(f"Best validation Macro-F1: {mlp_best_val_f1:.4f}")
print(f"Final feature count: {X_mlp_final_train.shape[1]}")
print("Full-train class frequencies:", full_train_class_counts.tolist())
print(f"Selection time: {mlp_selection_seconds:.1f} seconds")
print(f"Full-train refit time: {mlp_refit_seconds:.1f} seconds")
print(f"Final preprocessor saved to: {MLP_PREPROCESSOR_PATH}")
print(f"Final checkpoint saved to: {MLP_CHECKPOINT_PATH}")


## 7.3 MLP Held-Out Test Evaluation


In [ ]:
print("=" * 70)
print("PHASE 7.3: MLP HELD-OUT TEST EVALUATION")
print("=" * 70)

(
    mlp_test_metrics,
    mlp_classification_report,
    mlp_confusion_matrix,
    mlp_y_test_pred,
    mlp_y_test_proba,
) = evaluate_mlp(
    model=mlp_model,
    data_loader=mlp_test_loader,
    device=device,
)

mlp_test_metrics["Best Epoch"] = int(mlp_best_epoch)
mlp_test_metrics["Best Validation Macro F1"] = float(mlp_best_val_f1)
mlp_test_metrics["Selection Time (seconds)"] = round(
    float(mlp_selection_seconds), 2
)
mlp_test_metrics["Final Refit Time (seconds)"] = round(
    float(mlp_refit_seconds), 2
)
mlp_test_metrics["Training Time (seconds)"] = round(
    float(mlp_training_seconds), 2
)

mlp_classification_report.to_csv(
    MLP_REPORT_DIR / "mlp_classification_report.csv"
)
mlp_test_metrics_df = pd.DataFrame([mlp_test_metrics])
mlp_test_metrics_df.to_csv(
    MLP_REPORT_DIR / "mlp_test_metrics.csv",
    index=False,
)

print("\n--- MLP Held-Out Test Metrics ---")
display(mlp_test_metrics_df.round(4))
print("\n--- MLP Detailed Classification Report ---")
display(mlp_classification_report.round(4))

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(
    mlp_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cbar=True,
    ax=ax,
)
ax.set_title("MLP Confusion Matrix - Held-Out Test Set")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
fig.tight_layout()
fig.savefig(
    MLP_REPORT_DIR / "mlp_confusion_matrix.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 7.4 MLP Benchmark Against Traditional Models


In [ ]:
print("=" * 70)
print("PHASE 7.4: MLP BENCHMARK AGAINST TRADITIONAL MODELS")
print("=" * 70)

mlp_comparison_row = {
    "Model": "MLP",
    "Best Configuration": (
        f"{X_mlp_final_train.shape[1]}-128-64-32-3 | "
        "BatchNorm | ReLU | Dropout | Full-train refit"
    ),
    **{
        metric: mlp_test_metrics[metric]
        for metric in [
            "Accuracy",
            "Balanced Accuracy",
            "Macro F1",
            "Weighted F1",
            "Macro Precision",
            "Macro Recall",
            "<30 Precision",
            "<30 Recall",
            "<30 F1",
            "Macro ROC-AUC OvR",
            "Macro Average Precision OvR",
        ]
    },
}

if "master_comparison_df" not in locals():
    master_comparison_df = pd.read_csv(
        REPORTS_DIR / "master_model_comparison.csv"
    )

master_comparison_df = master_comparison_df[
    master_comparison_df["Model"] != "MLP"
].copy()
master_comparison_df = pd.concat(
    [master_comparison_df, pd.DataFrame([mlp_comparison_row])],
    ignore_index=True,
)
master_comparison_df = master_comparison_df.sort_values(
    by="Macro F1",
    ascending=False,
).reset_index(drop=True)

master_comparison_path = REPORTS_DIR / "master_model_comparison.csv"
master_comparison_df.to_csv(master_comparison_path, index=False)
display(master_comparison_df.round(4))

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))
palette_4 = {
    "Linear SVM": "#2b5c8f",
    "Logistic Regression": "#d95f02",
    "Random Forest": "#2ca02c",
    "MLP": "#7570b3",
}

metric_groups = [
    (
        ["Accuracy", "Balanced Accuracy", "Macro F1", "Weighted F1"],
        "Overall Performance Metrics (Test Set)",
        0.70,
    ),
    (
        ["<30 Precision", "<30 Recall", "<30 F1"],
        "Minority Class '<30' Readmission Metrics",
        0.70,
    ),
    (
        ["Macro ROC-AUC OvR", "Macro Average Precision OvR"],
        "Discrimination and Area Under Curve (OvR)",
        0.75,
    ),
]

for axis, (metrics, title, upper_limit) in zip(axes, metric_groups):
    plot_data = master_comparison_df.melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score",
    )
    sns.barplot(
        data=plot_data,
        x="Metric",
        y="Score",
        hue="Model",
        palette=palette_4,
        ax=axis,
    )
    axis.set_title(title, fontweight="bold")
    axis.set_ylim(0, upper_limit)
    axis.set_xlabel("")
    axis.set_ylabel("Score")
    axis.legend(loc="upper right", frameon=True, fontsize=9)
    for bar in axis.patches:
        height = bar.get_height()
        if not np.isnan(height) and height > 0:
            axis.annotate(
                f"{height:.3f}",
                (bar.get_x() + bar.get_width() / 2, height / 2),
                ha="center",
                va="center",
                fontsize=8,
                color="white",
                fontweight="bold",
                rotation=90,
            )

plt.tight_layout()
fig.savefig(
    MLP_REPORT_DIR / "mlp_four_model_comparison.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

rf_row = master_comparison_df.loc[
    master_comparison_df["Model"] == "Random Forest"
].iloc[0]
mlp_row = master_comparison_df.loc[
    master_comparison_df["Model"] == "MLP"
].iloc[0]

print("\n" + "=" * 70)
print("HEAD-TO-HEAD: FULL-TRAIN MLP VS RANDOM FOREST")
print("=" * 70)
print(
    f"Test Macro-F1: MLP={mlp_row['Macro F1']:.4f} | "
    f"RF={rf_row['Macro F1']:.4f}"
)
print(
    f"<30 Recall: MLP={mlp_row['<30 Recall']:.4f} | "
    f"RF={rf_row['<30 Recall']:.4f}"
)
print(
    f"Macro ROC-AUC OvR: MLP={mlp_row['Macro ROC-AUC OvR']:.4f} | "
    f"RF={rf_row['Macro ROC-AUC OvR']:.4f}"
)
print(f"Total MLP selection + refit time: {mlp_training_seconds:.1f}s")
print(f"Master comparison saved to: {master_comparison_path}")
